# Experiment 1 → added a layer BEFORE ResNet
               ResNet itself was untouched


In [1]:
import torch
import torchvision.models as models

# Load pretrained ResNet50
resnet = models.resnet50(weights='IMAGENET1K_V1')

# Count parameters
total = sum(p.numel() for p in resnet.parameters())
print(f"Total parameters: {total:,}")

# Look at first layer
print(f"\nFirst layer: {resnet.conv1}")

# Trace shapes through encoder
x = torch.randn(1, 3, 128, 128)  # our image size

with torch.no_grad():
    x1 = resnet.conv1(x)
    x1 = resnet.bn1(x1)
    x1 = resnet.relu(x1)
    x1 = resnet.maxpool(x1)
    
    x2 = resnet.layer1(x1)
    x3 = resnet.layer2(x2)
    x4 = resnet.layer3(x3)
    x5 = resnet.layer4(x4)

print("\n=== ResNet Shape Flow (128x128 input) ===")
print(f"After Conv1   : {x1.shape}")
print(f"After Layer1  : {x2.shape}")
print(f"After Layer2  : {x3.shape}")
print(f"After Layer3  : {x4.shape}")
print(f"After Layer4  : {x5.shape}")

Total parameters: 25,557,032

First layer: Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)

=== ResNet Shape Flow (128x128 input) ===
After Conv1   : torch.Size([1, 64, 32, 32])
After Layer1  : torch.Size([1, 256, 32, 32])
After Layer2  : torch.Size([1, 512, 16, 16])
After Layer3  : torch.Size([1, 1024, 8, 8])
After Layer4  : torch.Size([1, 2048, 4, 4])


In [2]:
import sys
sys.path.append("../src")

import torch
import importlib
import models.unet_prelayer as m1
importlib.reload(m1)

from models.unet_prelayer import UNetPreLayer

# Create model
model = UNetPreLayer(in_channels=12, out_channels=1)

# Count parameters
total  = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total:,}")

# Test with fake batch
fake   = torch.randn(2, 12, 128, 128)
output = model(fake)

print(f"Input  shape : {fake.shape}")
print(f"Output shape : {output.shape}")  # should be (2, 1, 128, 128)

Total parameters: 47,475,115
Input  shape : torch.Size([2, 12, 128, 128])
Output shape : torch.Size([2, 1, 128, 128])


In [3]:
model.eval()
x = torch.randn(1, 12, 128, 128)

with torch.no_grad():
    pre = model.pre_layer(x)
    e0  = model.encoder0(pre)
    e0p = model.pool(e0)
    e1  = model.encoder1(e0p)
    e2  = model.encoder2(e1)
    e3  = model.encoder3(e2)
    e4  = model.encoder4(e3)

print("=== Experiment 1 Shape Walkthrough ===\n")
print(f"Input          : {x.shape}")
print(f"After Pre-layer: {pre.shape}")
print(f"After Encoder0 : {e0.shape}")
print(f"After Layer1   : {e1.shape}")
print(f"After Layer2   : {e2.shape}")
print(f"After Layer3   : {e3.shape}")
print(f"After Layer4   : {e4.shape}")

=== Experiment 1 Shape Walkthrough ===

Input          : torch.Size([1, 12, 128, 128])
After Pre-layer: torch.Size([1, 3, 128, 128])
After Encoder0 : torch.Size([1, 64, 64, 64])
After Layer1   : torch.Size([1, 256, 32, 32])
After Layer2   : torch.Size([1, 512, 16, 16])
After Layer3   : torch.Size([1, 1024, 8, 8])
After Layer4   : torch.Size([1, 2048, 4, 4])


In [4]:
import sys
sys.path.append("../src")

import torch
import importlib
import train, visualize
importlib.reload(train)
importlib.reload(visualize)

from train import train as run_training
from models.unet_prelayer import UNetPreLayer
from preprocessing import load_stats
from dataset import get_dataloaders
import os

# ── Setup ──────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

IMAGE_DIR   = "../data/raw/images/"
MASK_DIR    = "../data/raw/masks/"
image_files = sorted(os.listdir(IMAGE_DIR))
stats       = load_stats("../data/processed/stats.json")

train_loader, val_loader, test_loader = get_dataloaders(
    IMAGE_DIR, MASK_DIR, image_files, stats, batch_size=16
)

# ── Model ──────────────────────────────────────
model_exp1 = UNetPreLayer(in_channels=12, out_channels=1).to(device)
print(f"Model ready ✅")

# ── Train ──────────────────────────────────────
train_losses, val_losses, metrics = run_training(
    model        = model_exp1,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    epochs       = 100,
    lr           = 0.001,
    save_dir     = "../outputs/checkpoints/exp1"
)

Device: cuda
  Stats loaded from ../data/processed/stats.json
Total   : 306
Train   : 214 (70%)
Val     : 45   (15%)
Test    : 47  (15%)
Dataset created with 214 samples
Dataset created with 45 samples
Dataset created with 47 samples
Model ready ✅
##############################################################
  Water Body Segmentation — Training
  Device : cuda  |  Seed : 42  |  Epochs : 100
  Params : 47,475,115  |  LR : 0.001  |  Batch : 16
##############################################################

  PHASE 1 / 3 — Data Exploration


C:\Users\POTATO\AppData\Roaming\Python\Python312\site-packages\rasterio\__init__.py:379: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


  >> Saved -> c:\Users\POTATO\Desktop\Cellula\Segmentation\Water-Body-Segmentation\outputs\plots\all_bands.png
  >> Saved -> c:\Users\POTATO\Desktop\Cellula\Segmentation\Water-Body-Segmentation\outputs\plots\image_vs_mask.png
  >> Saved -> c:\Users\POTATO\Desktop\Cellula\Segmentation\Water-Body-Segmentation\outputs\plots\band_distributions.png
  >> Saved -> c:\Users\POTATO\Desktop\Cellula\Segmentation\Water-Body-Segmentation\outputs\plots\first_batch.png
  >> Saved -> c:\Users\POTATO\Desktop\Cellula\Segmentation\Water-Body-Segmentation\outputs\plots\predictions_epoch_000.png

  PHASE 2 / 3 — Training


  Epoch 001/100 | Train: 1.1481 | Val: 2.1691 | IoU: 0.3745 | F1: 0.5445 | Time: 2s | LR: 1.00e-03
  [BEST] New best model saved! IoU=0.3745


  Epoch 002/100 | Train: 1.0484 | Val: 1.3408 | IoU: 0.5070 | F1: 0.6694 | Time: 2s | LR: 1.00e-03


KeyboardInterrupt: 

In [5]:
import importlib
import evaluate
importlib.reload(evaluate)

from evaluate import evaluate, evaluate_and_visualize

# Load best checkpoint correctly
best = UNetPreLayer(in_channels=12, out_channels=1).to(device)

checkpoint = torch.load(
    "../outputs/checkpoints/exp1/best_model.pth",
    map_location=device
)

# Reach inside the dictionary
best.load_state_dict(checkpoint["model_state_dict"])

print("=== EXPERIMENT 1 RESULTS ===")
exp1_metrics = evaluate(best, test_loader, device)
evaluate_and_visualize(
    best, test_loader, device,
    save_dir="../outputs/plots/exp1"
)

C:\Users\POTATO\AppData\Local\Temp\ipykernel_35628\1496605882.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(


RuntimeError: PytorchStreamReader failed locating file data/3: file not found

# Experiment 2 → go INSIDE ResNet
               surgically replace ONLY the first layer
               from Conv2d(3→64) to Conv2d(12→64)
               keep ALL other pretrained weights exactly

Experiment 1 → hired a translator to speak to the expert
Experiment 2 → taught the expert your language directly
```
Original ResNet first layer:
Conv2d(3, 64, kernel_size=7, stride=2, padding=3)
       ↑
       3 channels ← replace this

Our new first layer:
Conv2d(12, 64, kernel_size=7, stride=2, padding=3)
       ↑
       12 channels ✅

Everything else → identical pretrained weights

In [ ]:
import importlib
import models.unet_replace as m2
importlib.reload(m2)

from models.unet_replace import UNetReplace

model_exp2 = UNetReplace(in_channels=12, out_channels=1)

total  = sum(p.numel() for p in model_exp2.parameters())
print(f"\nTotal parameters: {total:,}")

# Test with fake batch
fake   = torch.randn(2, 12, 128, 128)
output = model_exp2(fake)

print(f"Input  shape : {fake.shape}")
print(f"Output shape : {output.shape}")

In [ ]:
# Confirm pretrained weights were copied correctly
resnet_original = models.resnet50(weights='IMAGENET1K_V1')
original_weights = resnet_original.conv1.weight  # (64, 3, 7, 7)

new_weights = model_exp2.encoder0[0].weight      # (64, 12, 7, 7)

# First 3 channels should match exactly
match = torch.allclose(
    new_weights[:, :3, :, :],
    original_weights,
    atol=1e-6
)

print(f"RGB weights copied correctly : {match}")
print(f"Original shape : {original_weights.shape}")
print(f"New layer shape: {new_weights.shape}")
print(f"\nChannel 0-2 mean : {new_weights[:,:3].abs().mean():.6f}  ← pretrained")
print(f"Channel 3-11 mean: {new_weights[:,3:].abs().mean():.6f}  ← initialized")

In [ ]:
model_exp2 = UNetReplace(in_channels=12, out_channels=1).to(device)

train_losses2, val_losses2, metrics2 = run_training(
    model        = model_exp2,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    epochs       = 50,
    lr           = 0.001,
    save_dir     = "../outputs/checkpoints/exp2"
)

In [ ]:
best2 = UNetReplace(in_channels=12, out_channels=1).to(device)

checkpoint2 = torch.load(
    "../outputs/checkpoints/exp2/best_model.pth",
    map_location=device
)
best2.load_state_dict(checkpoint2["model_state_dict"])

print("=== EXPERIMENT 2 RESULTS ===")
exp2_metrics = evaluate(best2, test_loader, device)
evaluate_and_visualize(
    best2, test_loader, device,
    save_dir="../outputs/plots/exp2"
)

# Experment 3
```
Experiment 1 & 2 → ResNet trained on ImageNet
                   learned from regular photos
                   cats, cars, buildings, people
                

Experiment 3     → model pretrained on SATELLITE data
                   already knows what satellite images look like
                   already understands spectral bands
                   already seen water, land, vegetation from above

In [6]:
import segmentation_models_pytorch as smp

# See what encoders are available
print("=== Some Available Encoders ===\n")

encoders = [
    "resnet50",
    "efficientnet-b4",
    "mit-b2",           # Mix Transformer — very powerful
    "mit-b4",
    "tu-maxvit_base_tf_512"
]

for enc in encoders:
    try:
        test_model = smp.Unet(
            encoder_name     = enc,
            encoder_weights  = None,
            in_channels      = 12,
            classes          = 1
        )
        params = sum(p.numel() for p in test_model.parameters())
        print(f"✅ {enc:35s} → {params:,} params")
    except Exception as e:
        print(f"❌ {enc:35s} → {str(e)[:50]}")

=== Some Available Encoders ===

✅ resnet50                            → 32,549,329 params
✅ efficientnet-b4                     → 20,229,577 params
❌ mit-b2                              → "Wrong encoder name `mit-b2`, supported encoders: 
❌ mit-b4                              → "Wrong encoder name `mit-b4`, supported encoders: 
✅ tu-maxvit_base_tf_512               → 122,649,029 params


In [9]:
import torch
import segmentation_models_pytorch as smp

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # ✅
try:
    model_exp3 = smp.Unet(
        encoder_name     = "mit_b2",         # ← underscore not hyphen
        encoder_weights  = "imagenet",
        in_channels      = 12,
        classes          = 1,
        activation       = None
    )

    model_exp3 = model_exp3.to(device)

    total  = sum(p.numel() for p in model_exp3.parameters())
    print(f"Total parameters: {total:,}")

    fake   = torch.randn(2, 12, 128, 128)
    output = model_exp3(fake.to(device))

    print(f"Input  shape : {fake.shape}")
    print(f"Output shape : {output.shape}")
except Exception as e:
    import traceback
    traceback.print_exc()


Total parameters: 27,505,233
Input  shape : torch.Size([2, 12, 128, 128])
Output shape : torch.Size([2, 1, 128, 128])


In [11]:
# Move to GPU for training
model_exp3 = smp.Unet(
    encoder_name    = "mit_b2",
    encoder_weights = "imagenet",
    in_channels     = 12,
    classes         = 1,
    activation      = None
).to(device)

print("Training Experiment 3 — MiT-B2\n")

train_losses3, val_losses3, metrics3 = run_training(
    model        = model_exp3,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    epochs       = 50,
    lr           = 0.0001,    # ← lower LR for transformer
    save_dir     = "../outputs/checkpoints/exp3"
)

Training Experiment 3 — MiT-B2

##############################################################
  Water Body Segmentation — Training
  Device : cuda  |  Seed : 42  |  Epochs : 50
  Params : 27,505,233  |  LR : 0.0001  |  Batch : 16
##############################################################

  PHASE 1 / 3 — Data Exploration


C:\Users\POTATO\AppData\Roaming\Python\Python312\site-packages\rasterio\__init__.py:379: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


  >> Saved -> c:\Users\POTATO\Desktop\Cellula\Segmentation\Water-Body-Segmentation\outputs\plots\all_bands.png
  >> Saved -> c:\Users\POTATO\Desktop\Cellula\Segmentation\Water-Body-Segmentation\outputs\plots\image_vs_mask.png
  >> Saved -> c:\Users\POTATO\Desktop\Cellula\Segmentation\Water-Body-Segmentation\outputs\plots\band_distributions.png
  >> Saved -> c:\Users\POTATO\Desktop\Cellula\Segmentation\Water-Body-Segmentation\outputs\plots\first_batch.png
  >> Saved -> c:\Users\POTATO\Desktop\Cellula\Segmentation\Water-Body-Segmentation\outputs\plots\predictions_epoch_000.png

  PHASE 2 / 3 — Training


  Epoch 001/50 | Train: 1.3724 | Val: 1.2322 | IoU: 0.4641 | F1: 0.6327 | Time: 2s | LR: 1.00e-04
  [BEST] New best model saved! IoU=0.4641


  Epoch 002/50 | Train: 1.1803 | Val: 1.0579 | IoU: 0.5700 | F1: 0.7252 | Time: 2s | LR: 1.00e-04
  [BEST] New best model saved! IoU=0.5700


...

  Epoch 046/50 | Train: 0.3662 | Val: 0.5209 | IoU: 0.8248 | F1: 0.9038 | Time: 2s | LR: 5.00e-05


  Epoch 047/50 | Train: 0.3676 | Val: 0.5244 | IoU: 0.8248 | F1: 0.9039 | Time: 2s | LR: 5.00e-05


  Epoch 048/50 | Train: 0.3723 | Val: 0.5292 | IoU: 0.8255 | F1: 0.9043 | Time: 2s | LR: 2.50e-05


  Epoch 049/50 | Train: 0.3707 | Val: 0.5153 | IoU: 0.8285 | F1: 0.9060 | Time: 2s | LR: 2.50e-05
  [BEST] New best model saved! IoU=0.8285


  Epoch 050/50 | Train: 0.3645 | Val: 0.5173 | IoU: 0.8297 | F1: 0.9068 | Time: 2s | LR: 2.50e-05
  [BEST] New best model saved! IoU=0.8297
  >> Saved -> c:\Users\POTATO\Desktop\Cellula\Segmentation\Water-Body-Segmentation\outputs\plots\predictions_epoch_050.png

  PHASE 3 / 3 — Post-Training Summary
--------------------------------------------------------------
  >> Best Epoch   : 50
  >> Best IoU     : 0.8297
  >> Final F1     : 0.9068
  >> Params       : 27,505,233
  >> Total Time   : 0h 1m 37s
  >> Time/Epoch   : 2.0s
  >> Early Stopped: False
--------------------------------------------------------------
  >> Saved -> c:\Users\POTATO\Desktop\Cellula\Segmentation\Water-Body-Segmentation\outputs\plots\loss_curves.png
  >> Saved -> c:\Users\POTATO\Desktop\Cellula\Segmentation\Water-Body-Segmentation\outputs\plots\metrics.png

  >> Results saved to: c:\Users\POTATO\Desktop\Cellula\Segmentation\Water-Body-Segmentation\outputs\results\results.json


In [12]:
# Evaluate Experiment 3 on test set first
best3 = smp.Unet(
    encoder_name    = "mit_b2",
    encoder_weights = None,
    in_channels     = 12,
    classes         = 1,
    activation      = None
).to(device)

checkpoint3 = torch.load(
    "../outputs/checkpoints/exp3/best_model.pth",
    map_location=device
)
best3.load_state_dict(checkpoint3["model_state_dict"])

print("=== EXPERIMENT 3 RESULTS ===")
exp3_metrics = evaluate(best3, test_loader, device)

C:\Users\POTATO\AppData\Local\Temp\ipykernel_35628\1470192710.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint3 = torch.load(


=== EXPERIMENT 3 RESULTS ===
  EVALUATION — Test Set
--------------------------------------------------------------
  FINAL TEST RESULTS
--------------------------------------------------------------
  >> IoU       : 0.8567
  >> F1        : 0.9219
  >> Precision : 0.9387
  >> Recall    : 0.9068
--------------------------------------------------------------
  >> Results saved to: c:\Users\POTATO\Desktop\Cellula\Segmentation\Water-Body-Segmentation\outputs\results\test_results.json
  >> Saved -> c:\Users\POTATO\Desktop\Cellula\Segmentation\Water-Body-Segmentation\outputs\plots\final_evaluation.png


In [ ]:
print("\n" + "="*65)
print("        FULL EXPERIMENT COMPARISON")
print("="*65)
print(f"{'Model':<20} | {'IoU':>6} | {'F1':>6} | {'Precision':>9} | {'Recall':>6}")
print("-"*65)

results = {
    "From Scratch"  : {"IoU": 0.8170, "F1": 0.8959,
                       "Precision": 0.9053, "Recall": 0.8879},
    "Exp1 PreLayer" : {"IoU": 0.8542, "F1": 0.9207,
                       "Precision": 0.9371, "Recall": 0.9061},
    "Exp2 Replace"  : {"IoU": 0.8154, "F1": 0.8971,
                       "Precision": 0.9367, "Recall": 0.8624},
    "Exp3 MiT-B2"   : exp3_metrics
}

for name, m in results.items():
    print(f"{name:<20} | {m['IoU']:>6.4f} | {m['F1']:>6.4f} | "
          f"{m['Precision']:>9.4f} | {m['Recall']:>6.4f}")

print("="*65)

best_name = max(results, key=lambda x: results[x]['IoU'])
print(f"\n🏆 Best Model : {best_name}")
print(f"   IoU        : {results[best_name]['IoU']:.4f}")


        FULL EXPERIMENT COMPARISON
Model                |    IoU |     F1 | Precision | Recall
-----------------------------------------------------------------
From Scratch         | 0.8170 | 0.8959 |    0.9053 | 0.8879
Exp1 PreLayer        | 0.8542 | 0.9207 |    0.9371 | 0.9061
Exp2 Replace         | 0.8154 | 0.8971 |    0.9367 | 0.8624
Exp3 MiT-B2          | 0.8567 | 0.9219 |    0.9387 | 0.9068

🏆 Best Model : Exp3 MiT-B2
   IoU        : 0.8567


: 